In [9]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ============================================================
# 1. SETTINGS
# ============================================================

hidden_dim = 32
encoding_dim = 8
batch_size = 64
learning_rate = 1e-3
epochs = 100
patience = 10
validation_split = 0.2

threshold_list = [0.002, 0.005, 0.003, 0.004]

train_cycle_fraction = 0.90

total_normal_samples = 50000
total_anomaly_samples = 10000

train_normal_samples = int(total_normal_samples * train_cycle_fraction)
test_normal_samples = total_normal_samples - train_normal_samples
test_anomaly_samples = total_anomaly_samples

n_runs = 5
seeds = [42, 43, 44, 45, 46]

data_path = Path("../data_processed/NASA/nasa_battery_features_rolling.csv")
results_root = Path("../results/AE_NASA_random_files_5runs")
results_root.mkdir(parents=True, exist_ok=True)

average_dir = results_root / "average_results"
average_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(data_path)

feature_cols = [
    "V", "I", "T",
    "Current_load", "Voltage_load", "Time",
    "V_diff1", "I_diff1", "T_diff1",
    "V_roll_mean", "V_roll_std", "V_roll_min", "V_roll_max",
    "I_roll_mean", "I_roll_std",
    "T_roll_mean", "T_roll_std"
]

required_cols = ["source_file", "y_true_file"] + feature_cols
df = df.dropna(subset=required_cols).reset_index(drop=True)

df["source_file_num"] = (
    df["source_file"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

df = df.sort_values(["source_file_num", "Time"]).reset_index(drop=True)

file_info = (
    df.groupby("source_file", as_index=False)
      .agg(
          source_file_num=("source_file_num", "first"),
          y_true_file=("y_true_file", "first"),
          n_rows=("source_file", "size")
      )
      .sort_values("source_file_num")
      .reset_index(drop=True)
)

normal_files_info = file_info[file_info["y_true_file"] == 0].reset_index(drop=True)
anomaly_files_info = file_info[file_info["y_true_file"] == 1].reset_index(drop=True)

print("Normal files total:", len(normal_files_info))
print("Anomaly files total:", len(anomaly_files_info))
print("Train normal samples:", train_normal_samples)
print("Test normal samples:", test_normal_samples)
print("Test anomaly samples:", test_anomaly_samples)

# ============================================================
# 3. HELPER
# ============================================================

def safe_name(value):
    return str(value).replace(".", "_").replace("-", "minus_")


def build_autoencoder(input_dim, hidden_dim, encoding_dim, learning_rate):
    inputs = Input(shape=(input_dim,))
    x = Dense(hidden_dim, activation="relu")(inputs)
    latent = Dense(encoding_dim, activation="relu")(x)
    x = Dense(hidden_dim, activation="relu")(latent)
    outputs = Dense(input_dim, activation="linear")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model


all_results = []

# ============================================================
# 4. 5 RANDOM RUNS BY SOURCE_FILE
# ============================================================

for run_id, seed in enumerate(seeds, start=1):
    print("\n================================================")
    print(f"RUN {run_id}/{n_runs}, seed = {seed}")
    print("================================================")

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    run_dir = results_root / f"run_{run_id}_seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    normal_files_shuffled = normal_files_info.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    split_normal_idx = int(len(normal_files_shuffled) * train_cycle_fraction)

    train_normal_files = normal_files_shuffled.iloc[:split_normal_idx]["source_file"]
    test_normal_files = normal_files_shuffled.iloc[split_normal_idx:]["source_file"]

    anomaly_files_shuffled = anomaly_files_info.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    test_anomaly_files = anomaly_files_shuffled["source_file"]

    train_window = df[df["source_file"].isin(train_normal_files)].copy()
    test_normal_window = df[df["source_file"].isin(test_normal_files)].copy()
    test_anomaly_window = df[df["source_file"].isin(test_anomaly_files)].copy()

    train_normal = (
        train_window[train_window["y_true_file"] == 0]
        .iloc[:train_normal_samples]
        .copy()
    )

    test_normal = (
        test_normal_window[test_normal_window["y_true_file"] == 0]
        .iloc[:test_normal_samples]
        .copy()
    )

    test_anomaly = (
        test_anomaly_window[test_anomaly_window["y_true_file"] == 1]
        .iloc[:test_anomaly_samples]
        .copy()
    )

    test_data = pd.concat(
        [test_normal, test_anomaly],
        ignore_index=True
    )

    X_train_normal_raw = train_normal[feature_cols].copy()
    X_test_raw = test_data[feature_cols].copy()
    y_test = test_data["y_true_file"].astype(int).copy()

    print("Split type: random split by complete source_file cycles")
    print("Train normal files:", train_normal["source_file"].nunique())
    print("Test normal files:", test_normal["source_file"].nunique())
    print("Test anomaly files:", test_anomaly["source_file"].nunique())
    print("Train normal rows:", len(train_normal))
    print("Test normal rows:", len(test_normal))
    print("Test anomaly rows:", len(test_anomaly))

    scaler = StandardScaler()
    X_train_normal = scaler.fit_transform(X_train_normal_raw)
    X_test = scaler.transform(X_test_raw)

    model = build_autoencoder(
        input_dim=X_train_normal.shape[1],
        hidden_dim=hidden_dim,
        encoding_dim=encoding_dim,
        learning_rate=learning_rate
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_normal,
        X_train_normal,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        shuffle=True,
        callbacks=[early_stopping],
        verbose=0
    )

    test_recon = model.predict(X_test, verbose=0)
    reconstruction_error = np.mean(np.square(X_test - test_recon), axis=1)

    print("Reconstruction error min:", reconstruction_error.min())
    print("Reconstruction error max:", reconstruction_error.max())
    print("Reconstruction error mean:", reconstruction_error.mean())

    for threshold in threshold_list:
        threshold_dir = run_dir / f"threshold_{safe_name(threshold)}"
        threshold_dir.mkdir(parents=True, exist_ok=True)

        y_pred = (reconstruction_error > threshold).astype(int)

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        exp_name = (
            f"AE_NASA_run_{run_id}_seed_{seed}_"
            f"split_{safe_name(train_cycle_fraction)}_"
            f"threshold_{safe_name(threshold)}"
        )

        df_results = test_data.copy()
        df_results["reconstruction_error"] = reconstruction_error
        df_results["threshold"] = threshold
        df_results["is_anomaly"] = y_pred
        df_results.to_csv(threshold_dir / "predictions.csv", index=False)

        print("--------------------------------------")
        print(exp_name)
        print("--------------------------------------")
        print(f"Threshold: {threshold}")
        print(f"Accuracy : {accuracy:.2f}%")
        print(f"Precision: {precision:.2f}%")
        print(f"Recall   : {recall:.2f}%")
        print(f"F1-score : {f1:.2f}%")
        print("Confusion Matrix")
        print(cm)

        all_results.append({
            "run_id": run_id,
            "seed": seed,
            "train_cycle_fraction": train_cycle_fraction,
            "threshold": threshold,
            "hidden_dim": hidden_dim,
            "encoding_dim": encoding_dim,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "epochs": epochs,
            "patience": patience,
            "epochs_trained": len(history.history["loss"]),
            "best_val_loss": round(float(np.min(history.history["val_loss"])), 8),
            "final_train_loss": round(float(history.history["loss"][-1]), 8),
            "train_normal_samples": len(train_normal),
            "test_normal_samples": len(test_normal),
            "test_anomaly_samples": len(test_anomaly),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        })

        # ========================================================
        # Metrics table
        # ========================================================

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu Autoencoder – NASA Battery", fontsize=12, pad=20)
        plt.savefig(threshold_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ========================================================
        # Confusion matrix
        # ========================================================

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        cm_threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > cm_threshold else "white"
                plt.text(
                    j, i, cm[i, j],
                    ha="center",
                    va="center",
                    color=color
                )

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(threshold_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ========================================================
        # Anomaly plot
        # ========================================================

        x_axis = np.arange(len(df_results))

        plt.figure(figsize=(14, 5))
        plt.plot(x_axis, df_results["V"], linewidth=1)

        anomaly_idx = df_results["is_anomaly"] == 1
        plt.scatter(
            np.array(x_axis)[anomaly_idx],
            df_results.loc[anomaly_idx, "V"],
            s=18
        )

        plt.title(exp_name)
        plt.xlabel("Index")
        plt.ylabel("V")
        plt.tight_layout()
        plt.savefig(threshold_dir / "anomaly_plot.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ========================================================
        # Reconstruction error histogram with log scale
        # ========================================================

        normal_scores = reconstruction_error[y_test.values == 0]
        anomaly_scores = reconstruction_error[y_test.values == 1]

        plt.figure(figsize=(18, 6))

        min_score = max(float(reconstruction_error.min()), 1e-8)
        max_score = float(reconstruction_error.max())

        bins = np.logspace(
            np.log10(min_score),
            np.log10(max_score),
            60
        )

        plt.hist(normal_scores, bins=bins, alpha=0.45, label="Normálne")
        plt.hist(anomaly_scores, bins=bins, alpha=0.55, label="Anomálie")

        plt.axvline(
            threshold,
            linestyle="--",
            linewidth=2,
            label=f"Threshold = {threshold}"
        )

        plt.xscale("log")

        x_right = np.percentile(reconstruction_error, 99.5)
        x_right = max(x_right, threshold * 1.5, min_score * 10)
        plt.xlim(min_score, x_right)

        plt.title("Histogram rekonštrukčnej chyby Autoencoder – NASA Battery")
        plt.xlabel("Rekonštrukčná chyba (log škála)")
        plt.ylabel("Počet vzoriek")
        plt.legend()
        plt.grid(True, which="both")
        plt.tight_layout()
        plt.savefig(threshold_dir / "reconstruction_error_histogram.png", dpi=300, bbox_inches="tight")
        plt.close()

# ============================================================
# 5. SAVE SUMMARY RESULTS
# ============================================================

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results_all_runs.csv", index=False)

metric_cols = [
    "accuracy_%", "precision_%", "recall_%", "f1_%",
    "tn", "fp", "fn", "tp"
]

average_results = (
    results_df
    .groupby(["train_cycle_fraction", "threshold"], as_index=False)[metric_cols]
    .mean(numeric_only=True)
)

for col in ["accuracy_%", "precision_%", "recall_%", "f1_%"]:
    average_results[col] = average_results[col].round(2)

for col in ["tn", "fp", "fn", "tp"]:
    average_results[col] = average_results[col].round(1)

average_results = average_results.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
average_results.to_csv(average_dir / "average_results_5_runs.csv", index=False)

# ============================================================
# 6. CREATE SEPARATE AVERAGE FOLDER FOR EACH THRESHOLD
# ============================================================

for _, avg_row in average_results.iterrows():

    threshold = avg_row["threshold"]

    threshold_average_dir = (
        average_dir / f"threshold_{safe_name(threshold)}"
    )
    threshold_average_dir.mkdir(parents=True, exist_ok=True)

    threshold_average_df = pd.DataFrame([avg_row])
    threshold_average_df.to_csv(
        threshold_average_dir / "average_result_5_runs.csv",
        index=False
    )

    # ========================================================
    # Average metrics table for this threshold
    # ========================================================

    metrics_average_df = pd.DataFrame({
        "Metrika": [
            "Presnosť (Accuracy)",
            "Precíznosť (Precision)",
            "Citlivosť (Recall)",
            "F1-skóre"
        ],
        "Priemerná hodnota (%)": [
            avg_row["accuracy_%"],
            avg_row["precision_%"],
            avg_row["recall_%"],
            avg_row["f1_%"]
        ]
    })

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")

    table = ax.table(
        cellText=metrics_average_df.values,
        colLabels=metrics_average_df.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#dce6f1")
        else:
            cell.set_facecolor("#f9f9f9")

    plt.title(
        f"Priemerné metriky modelu Autoencoder – NASA Battery\n"
        f"5 behov, threshold = {threshold}",
        fontsize=12,
        pad=20
    )

    plt.savefig(
        threshold_average_dir / "average_metrics_table.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    # ========================================================
    # Average confusion matrix for this threshold
    # ========================================================

    avg_cm = np.array([
        [avg_row["tn"], avg_row["fp"]],
        [avg_row["fn"], avg_row["tp"]]
    ])

    plt.figure(figsize=(6, 5))
    plt.imshow(avg_cm, interpolation="nearest")
    plt.title(
        f"Priemerná matica zámen Autoencoder – NASA Battery\n"
        f"Threshold = {threshold}"
    )
    plt.colorbar()

    classes = ["Normálne", "Anomálie"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    cm_threshold = avg_cm.max() / 2.0 if avg_cm.max() > 0 else 0.5

    for i in range(avg_cm.shape[0]):
        for j in range(avg_cm.shape[1]):
            color = "black" if avg_cm[i, j] > cm_threshold else "white"
            plt.text(
                j, i, avg_cm[i, j],
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()

    plt.savefig(
        threshold_average_dir / "average_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

# ============================================================
# 7. SAVE BEST AVERAGE RESULT
# ============================================================

best_average = average_results.iloc[[0]].copy()
best_average.to_csv(average_dir / "best_average_result.csv", index=False)

print("Saved:")
print(results_root / "summary_results_all_runs.csv")
print(average_dir / "average_results_5_runs.csv")
print(average_dir / "best_average_result.csv")

for threshold in threshold_list:
    print(average_dir / f"threshold_{safe_name(threshold)}")

print("Done.")

Normal files total: 2215
Anomaly files total: 517
Train normal samples: 45000
Test normal samples: 5000
Test anomaly samples: 10000

RUN 1/5, seed = 42
Split type: random split by complete source_file cycles
Train normal files: 148
Test normal files: 18
Test anomaly files: 33
Train normal rows: 45000
Test normal rows: 5000
Test anomaly rows: 10000
Reconstruction error min: 4.514954857022453e-05
Reconstruction error max: 1454.575686648458
Reconstruction error mean: 0.15294239054871223
--------------------------------------
AE_NASA_run_1_seed_42_split_0_9_threshold_0_002
--------------------------------------
Threshold: 0.002
Accuracy : 77.24%
Precision: 76.04%
Recall   : 96.17%
F1-score : 84.93%
Confusion Matrix
[[1969 3031]
 [ 383 9617]]
--------------------------------------
AE_NASA_run_1_seed_42_split_0_9_threshold_0_005
--------------------------------------
Threshold: 0.005
Accuracy : 75.47%
Precision: 84.08%
Recall   : 77.97%
F1-score : 80.91%
Confusion Matrix
[[3524 1476]
 [2203 